In [2]:
import json
import sys
import os
import time

import pandas as pd

from mods_old_reddit import sanitize_filename

sys.path.append(".")
import dimen_generation

In [3]:
# Load community embedding
vectors, metadata = dimen_generation.load_embedding()

In [4]:
df=pd.read_csv("subredditDataWA.csv", index_col=0)

In [5]:
def get_founder(row):
    subreddit_name = row.subreddit_name
    subreddit_start=row.subreddit_creation_date
    fpath = f'mods_json/{sanitize_filename(subreddit_name)}_mods.json'
    if os.path.exists(fpath):
        with open(fpath, "r", encoding="utf-8") as f:
            json_data = json.load(f)
        for mod in json_data.get("data", dict()).get("children", []):
            mod_start = mod.get("date", time.time())
            if mod_start<=subreddit_start+1000*60*60*1: #1h since creation
                return mod.get("name")

In [6]:
df["founder"]=df.apply(get_founder, axis=1)

In [7]:
df.dropna(subset=["founder"], inplace=True)

In [8]:
df

,subreddit_name,earliest_comment,earliest_post,subreddit_creation_date,founder
0,BMW,1.262818e+09,1222892660,1222892589,joka86
4,lookingforheroes,1.435388e+09,1435385158,1435384268,dirtymonkey
7,Boardgamedeals,1.385817e+09,1381219040,1381218676,OpT1mUs
10,BoardGameExchange,1.397569e+09,1397496219,1397495617,whatsupraleigh
12,uncensorship,1.333574e+09,1333557414,1331779436,sunshine-x
...,...,...,...,...,...
8963,umass,1.286858e+09,1227249394,1227249345,xnoseatbelt
8966,longrange,1.332593e+09,1332584222,1332583657,pestilence
8970,longstabbything,1.385718e+09,1366602890,1366602774,chooksta
8974,lookatmydog,1.290031e+09,1289880079,1289879442,rockon4life45


In [9]:
# Compute all pairs of similar communities
dimen_generator = dimen_generation.DimenGenerator(vectors)

100060 valid directions, 100060 calculated.


In [10]:

anti_ct=["QAnonCasualties", "ReQovery", "skeptic", "DebunkThis", "conspiracytheories", "actualconspiracies", "ConspiracyPsychology", "conspiratard", ]

In [11]:
q_subs=["greatawakening", "CBTS_Stream"]

In [12]:
anti_q=["ReQovery", "QAnonCasualties", "Qult_Headquarters"]

In [13]:
for i in anti_ct+q_subs+anti_q:
    if i in metadata.index:
        print(i)

skeptic
DebunkThis
actualconspiracies
conspiratard
CBTS_Stream


In [14]:
# Generate dimensions through seed augmentation
seeds=[
    ("conspiracy", "skeptic"),
    ("conspiracy", "DebunkThis"),
    ("conspiracy", "actualconspiracies"),
    ("conspiracy", "conspiratard"),
        ("conspiracy", "science"),

    # ("conspiracy", "AskMenOver30"),
    # ("daddit", "Mommit"),
    # ("hillaryclinton", "The_Donald"),
    # ("nyc", "nycmeetups"),
    # ("memes", "ImGoingToHellForThis"),
    # ("PS3", "PS4")
]
dimensions = dimen_generator.generate_dimensions_from_seeds(seeds)

In [15]:
# Print which seeds were found
dimen_names = [i[1] for i in seeds]

for name, dimen in zip(dimen_names, dimensions):
    print("Dimension %s:" % name)
    print("\tSeed: %s" % dimen["seed"])
    print("\tFound seeds:")
    for c1, c2 in zip(dimen["left_comms"], dimen["right_comms"]):
        print("\t\t%s -> %s" % (c1, c2))

Dimension skeptic:
	Seed: [('conspiracy', 'skeptic')]
	Found seeds:
		conspiracy -> skeptic
		worldpolitics -> Freethought
		Mcat -> biology
		GasTheSnoo -> ThePopcornStand
		TrayvonMartin -> antisrs
		metarage -> aaaaaatheismmmmmmmmmm
		pantheism -> TrueAtheism
		TopMindsOfReddit -> badscience
		worldnews -> science
		CigarMarket -> Scotch
Dimension DebunkThis:
	Seed: [('conspiracy', 'DebunkThis')]
	Found seeds:
		conspiracy -> DebunkThis
		911truth -> skeptic
		worldpolitics -> Freethought
		mcrealmsservers -> redstone
		GhostAdventures -> UrbanMyths
		pantheism -> TrueAtheism
		almosthomeless -> internetparents
		The_Brendan -> KenM
		fantasyfootball -> FFCommish
		metarage -> aaaaaatheismmmmmmmmmm
Dimension actualconspiracies:
	Seed: [('conspiracy', 'actualconspiracies')]
	Found seeds:
		conspiracy -> actualconspiracies
		911truth -> skeptic
		angryjoeshow -> JimSterling
		MensRights -> INeedFeminismBecause
		Bible -> AcademicBiblical
		wwesupercard -> MarvelPuzzleQuest
		insects -

In [16]:
# Calculate scores for communities
scores = dimen_generation.score_embedding(vectors, zip(dimen_names, dimensions))
scores.head(5)

,skeptic,DebunkThis,actualconspiracies,conspiratard,science
community,,,,,
keto,0.062866,0.006791,-0.036743,-0.173984,0.119805
AskReddit,0.059938,0.043215,-0.172344,-0.114915,0.240253
funny,0.063697,0.030647,-0.191854,-0.140396,0.307331
politics,-0.090397,-0.145595,-0.108447,-0.139652,0.099342
pics,0.052120,-0.058290,-0.226880,-0.213906,0.333191


In [69]:
# Save result (compare with scores.csv from repository)
scores.to_csv('scores.csv')

In [78]:
scores.sort_values("DebunkThis")

,skeptic,DebunkThis,actualconspiracies,conspiratard,science
community,,,,,
conspiracy,-0.309361,-0.343974,-0.360720,-0.358077,-0.316146
The_Brendan,-0.326868,-0.310640,-0.141163,-0.065966,-0.233154
GhostAdventures,-0.189492,-0.293654,-0.296660,-0.241721,-0.289401
The_Donald,-0.329621,-0.281312,-0.275586,-0.220053,-0.230060
Infowars,-0.278236,-0.266452,-0.330786,-0.241433,-0.319274
...,...,...,...,...,...
TheFacebookDelusion,0.342234,0.466722,0.174183,0.253793,0.012512
aaaaaatheismmmmmmmmmm,0.454200,0.476968,0.216233,0.208856,0.143172
atheistparents,0.462975,0.478903,0.305876,0.041205,0.266185


In [17]:
df["subreddit_name_lower"]=df.subreddit_name.str.lower()

In [18]:
scores["subreddit_name_lower"]=scores.index.str.lower()

In [19]:
df_scored=pd.merge(df, scores, on="subreddit_name_lower", how="left").sort_values(
    "DebunkThis", ascending=True
)

In [20]:
df_scored.sort_values("science")

,subreddit_name,earliest_comment,earliest_post,subreddit_creation_date,founder,subreddit_name_lower,skeptic,DebunkThis,actualconspiracies,conspiratard,science
1336,HillaryMeltdown,1.478668e+09,1478666037,1478664555,Trumpropriation,hillarymeltdown,-0.250956,-0.163997,-0.161766,-0.072136,-0.296508
342,McKaylaMaroney,1.344813e+09,1344471214,1343871129,nivanbotemill,mckaylamaroney,-0.212374,-0.111376,-0.142247,-0.103595,-0.295630
659,GhostAdventures,1.328202e+09,1326418758,1295067051,Rachet20,ghostadventures,-0.189492,-0.293654,-0.296660,-0.241721,-0.289401
1105,ConspiracyII,1.493125e+09,1493119972,1493119405,qwertyqyle,conspiracyii,-0.151623,-0.172465,-0.095091,-0.189918,-0.286953
620,whiteknighting,1.356720e+09,1356674504,1356673921,thedeafpoliceman,whiteknighting,-0.090877,-0.023857,-0.038573,0.152965,-0.272983
...,...,...,...,...,...,...,...,...,...,...,...
2914,Angers,1.721740e+09,1719077605,1719077295,Ken_kitano,angers,NaN,NaN,NaN,NaN,NaN
2918,antgrumps,1.606403e+09,1473000296,1473000125,kianworld,antgrumps,NaN,NaN,NaN,NaN,NaN
2952,ATS,1.295352e+09,1294527261,1294527208,doublec,ats,NaN,NaN,NaN,NaN,NaN
2954,avens,1.533136e+09,1359921361,1359916216,Rabid_Snowman,avens,NaN,NaN,NaN,NaN,NaN


In [21]:
df_scored.sort_values("skeptic")


,subreddit_name,earliest_comment,earliest_post,subreddit_creation_date,founder,subreddit_name_lower,skeptic,DebunkThis,actualconspiracies,conspiratard,science
1444,DACA,1.448306e+09,1443338468,1418841067,Crunketh,daca,-0.262949,-0.163808,-0.204319,-0.156633,-0.097530
637,wholesomebpt,1.482956e+09,1482956113,1482955899,Zetice,wholesomebpt,-0.262375,-0.166481,-0.102000,-0.042589,-0.034231
610,WhereIsAssange,1.477162e+09,1477111794,1477111440,ThoriumWL,whereisassange,-0.257163,-0.218543,-0.159982,-0.283750,-0.173647
1870,PanamaPapers,1.459710e+09,1459708177,1459708100,omri100,panamapapers,-0.254611,-0.171879,-0.062707,-0.153222,-0.017990
2605,LatinoPeopleTwitter,1.436400e+09,1436399584,1436385168,Isai76,latinopeopletwitter,-0.253124,-0.149950,-0.038764,0.013166,-0.169704
...,...,...,...,...,...,...,...,...,...,...,...
2914,Angers,1.721740e+09,1719077605,1719077295,Ken_kitano,angers,NaN,NaN,NaN,NaN,NaN
2918,antgrumps,1.606403e+09,1473000296,1473000125,kianworld,antgrumps,NaN,NaN,NaN,NaN,NaN
2952,ATS,1.295352e+09,1294527261,1294527208,doublec,ats,NaN,NaN,NaN,NaN,NaN
2954,avens,1.533136e+09,1359921361,1359916216,Rabid_Snowman,avens,NaN,NaN,NaN,NaN,NaN


In [88]:
df_scored.sort_values("science").to_csv("subredditDataWAScored.csv", index=False)

In [23]:
df_scored.subreddit_name.values

<StringArray>
[    'GhostAdventures',       'XboxOneGamers',       'GothBoiClique',
      'WhereIsAssange',      'Cod4Remastered',       'AutoNewspaper',
              'Bumble',          'The_Dennis',        'fyrefestival',
           'iBUYPOWER',
 ...
                'adio',                 'Age',              'aleigh',
                'amen', 'andomActsOfKindness',              'Angers',
           'antgrumps',                 'ATS',               'avens',
             'awdenim']
Length: 2995, dtype: str